# Stage 6D — Strategy–Result Linkage Eligibility Screening

This notebook screens whether governed Stage 6C evidence supports a defensible strategy–result linkage candidate.

All 22 documented actions are screened against the 16 frozen Stage 5 linkage gates. Company-reported explanations remain separate from independent evidence, inherited Stage 4 portfolio findings remain context unless scope alignment is explicit, and missing compatible outcomes are preserved rather than replaced.

Stage 6D does not estimate causal effects, rank strategy effectiveness, construct a composite score, or revise the Stage 4 overall-winner conclusion.


## Environment Setup

Lock the authoritative Stage 6C commit and the Stage 4–6C evidence and governance files required for linkage screening.


In [1]:
from __future__ import annotations

import hashlib
import os
import shutil
import urllib.error
import urllib.parse
import urllib.request
from pathlib import Path

import pandas as pd

REPOSITORY_FULL_NAME = "Ronaldo-spec/indonesia-fmcg-brand-portfolio-analysis"
INPUT_COMMIT = "87009a9583242bd0138de2633a49c8d8d9f3fc41"

INPUT_LOCKS = {
    "data/analytical/stage6c_strategy_actions.csv":
        "092a12a92f982a93101ea31e049f1c2292cedf54aa7decd157952bec185f3822",
    "data/analytical/stage6c_result_observations.csv":
        "bc91bd300a01d3eecbc7c7b81d60747f203a4a63aff39f890aaffafc67ab9ad1",
    "data/analytical/stage6c_company_attribution_claims.csv":
        "44aaf59c2bd454b486b48ca8eb0f762cc19124f55cafea408bb7a44460b6f2e1",
    "metadata/stage6c_extraction_exceptions.csv":
        "a0dbfcbfcc41445d5fdf3b866fe32ff9f0372f40656ef211ff2a9dafb725ebb4",
    "metadata/stage6c_extraction_validation.csv":
        "c606581f73193aa24f00cbe1140c8b1ee1403727247ce7b7d4cee4cdef552e7f",
    "data/analytical/stage4_final_findings.csv":
        "847d17369b60f950495467a9ecedbbbd2bd5397d49aa6a845be6dd784b8ee35c",
    "metadata/stage5_linkage_eligibility_rules.csv":
        "39142fd09351f473c033a8862678e5ff1327871407e05a8b1e6b9093c74b092a",
    "metadata/stage5_research_questions.csv":
        "6f2854d93713534e89f64ff743d4ec2a0bdd101e9a109989cb13bfcecd1aaa14",
    "metadata/stage5_attribution_evidence_rules.csv":
        "a2afb7e170f71d6bb6937cfc41c3e84c027c4e2df82525096a2cf776215006a5",
    "metadata/stage5_period_scope.csv":
        "7c70d6e7c6b1f917ad3685c08f5f0b2d215e7ac62b008b5e7cee0bfec0f3b713",
    "metadata/stage5_strategy_taxonomy.csv":
        "65c5ffd83c847b9f93c64a4bafd8ca6bf27abe26ee5ea2acd2ff69e07862c684",
    "metadata/stage5_outcome_taxonomy.csv":
        "32a3b729ea1e903a24a30e5927e47e909745e527c42281dd0997a7919320cc03",
}

OUTPUT_ROOT = Path(
    os.environ.get("FMCG_STAGE6D_OUTPUT_ROOT", "/content/fmcg_stage6d_outputs")
)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 160)

print(f"Locked Stage 6C commit: {INPUT_COMMIT}")
print(f"Required governed inputs: {len(INPUT_LOCKS)}")
print(f"Output root: {OUTPUT_ROOT}")


Locked Stage 6C commit: 87009a9583242bd0138de2633a49c8d8d9f3fc41
Required governed inputs: 12
Output root: /content/fmcg_stage6d_outputs


## Locked Input Retrieval

Retrieve only the 12 governed inputs from the authoritative Stage 6C commit. Colab uses `GITHUB_TOKEN`; local validation may use `FMCG_STAGE6D_INPUT_ROOT`.


In [2]:
configured_root = os.environ.get("FMCG_STAGE6D_INPUT_ROOT")

if configured_root:
    INPUT_ROOT = Path(configured_root)
    input_mode = "local_validation_root"
else:
    try:
        from google.colab import userdata
    except ImportError as exc:
        raise RuntimeError(
            "Run in Google Colab or set FMCG_STAGE6D_INPUT_ROOT."
        ) from exc

    github_token = userdata.get("GITHUB_TOKEN")
    if not github_token:
        raise RuntimeError("Colab Secret GITHUB_TOKEN is unavailable.")

    INPUT_ROOT = Path("/content/fmcg_stage6d_inputs")
    if INPUT_ROOT.exists():
        shutil.rmtree(INPUT_ROOT)
    INPUT_ROOT.mkdir(parents=True, exist_ok=True)

    for relative_path in INPUT_LOCKS:
        encoded_path = urllib.parse.quote(relative_path, safe="/")
        url = (
            f"https://api.github.com/repos/{REPOSITORY_FULL_NAME}"
            f"/contents/{encoded_path}?ref={INPUT_COMMIT}"
        )
        request = urllib.request.Request(
            url,
            headers={
                "Authorization": f"Bearer {github_token}",
                "Accept": "application/vnd.github.raw+json",
                "X-GitHub-Api-Version": "2022-11-28",
                "User-Agent": "fmcg-stage6d-colab",
            },
        )
        destination = INPUT_ROOT / relative_path
        destination.parent.mkdir(parents=True, exist_ok=True)
        try:
            with urllib.request.urlopen(request, timeout=60) as response:
                destination.write_bytes(response.read())
        except urllib.error.HTTPError as exc:
            raise RuntimeError(
                f"Input retrieval failed for {relative_path}: HTTP {exc.code}"
            ) from exc

    del github_token
    input_mode = "locked_github_commit"

missing = [
    p for p in INPUT_LOCKS if not (INPUT_ROOT / p).exists()
]
if missing:
    raise FileNotFoundError(f"Missing Stage 6D inputs: {missing}")

print(f"Input mode: {input_mode}")
print(f"Required files found: {len(INPUT_LOCKS)}/{len(INPUT_LOCKS)}")


Input mode: locked_github_commit
Required files found: 12/12


## Input Integrity and Frozen Framework

Verify locked SHA-256 values, confirm the Stage 6C gate, and validate the inherited action, result, claim, Stage 4 finding, and frozen Stage 5 governance registries.


In [3]:
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

lock_rows = []
for relative_path, expected in INPUT_LOCKS.items():
    actual = sha256_file(INPUT_ROOT / relative_path)
    lock_rows.append({
        "file_path": relative_path,
        "expected_sha256": expected,
        "actual_sha256": actual,
        "hash_match": actual == expected,
        "locked_repository_commit": INPUT_COMMIT,
    })

stage6d_input_lock = pd.DataFrame(lock_rows)
if not stage6d_input_lock["hash_match"].all():
    failed = stage6d_input_lock.loc[
        ~stage6d_input_lock["hash_match"], "file_path"
    ].tolist()
    raise RuntimeError(f"Input checksum failure: {failed}")

actions = pd.read_csv(
    INPUT_ROOT / "data/analytical/stage6c_strategy_actions.csv",
    dtype=str, keep_default_na=False,
)
results = pd.read_csv(
    INPUT_ROOT / "data/analytical/stage6c_result_observations.csv",
    dtype=str, keep_default_na=False,
)
company_claims = pd.read_csv(
    INPUT_ROOT / "data/analytical/stage6c_company_attribution_claims.csv",
    dtype=str, keep_default_na=False,
)
stage6c_exceptions = pd.read_csv(
    INPUT_ROOT / "metadata/stage6c_extraction_exceptions.csv",
    dtype=str, keep_default_na=False,
)
stage6c_validation = pd.read_csv(
    INPUT_ROOT / "metadata/stage6c_extraction_validation.csv",
    dtype=str, keep_default_na=False,
)
stage4_findings = pd.read_csv(
    INPUT_ROOT / "data/analytical/stage4_final_findings.csv",
    dtype=str, keep_default_na=False,
)
linkage_rules = pd.read_csv(
    INPUT_ROOT / "metadata/stage5_linkage_eligibility_rules.csv",
    dtype=str, keep_default_na=False,
)
research_questions = pd.read_csv(
    INPUT_ROOT / "metadata/stage5_research_questions.csv",
    dtype=str, keep_default_na=False,
)
attribution_rules = pd.read_csv(
    INPUT_ROOT / "metadata/stage5_attribution_evidence_rules.csv",
    dtype=str, keep_default_na=False,
)
period_scope = pd.read_csv(
    INPUT_ROOT / "metadata/stage5_period_scope.csv",
    dtype=str, keep_default_na=False,
)
strategy_taxonomy = pd.read_csv(
    INPUT_ROOT / "metadata/stage5_strategy_taxonomy.csv",
    dtype=str, keep_default_na=False,
)
outcome_taxonomy = pd.read_csv(
    INPUT_ROOT / "metadata/stage5_outcome_taxonomy.csv",
    dtype=str, keep_default_na=False,
)

final_stage6c = stage6c_validation.loc[
    stage6c_validation["check_id"] == "S6C038"
].iloc[0]

if (
    final_stage6c["result"] != "PASS_WITH_CAVEAT"
    or final_stage6c["status"] != "passed_with_caveat"
):
    raise RuntimeError("Unexpected Stage 6C final gate.")

expected_sets = [
    (set(linkage_rules["linkage_rule_id"]), {f"LNK{i:02d}" for i in range(1,17)}, "linkage rules"),
    (set(research_questions["question_id"]), {f"SRQ{i:02d}" for i in range(1,12)}, "research questions"),
    (set(attribution_rules["attribution_rule_id"]), {f"ATTR{i:02d}" for i in range(1,10)}, "attribution rules"),
    (set(period_scope["period_rule_id"]), {f"PER{i:02d}" for i in range(1,8)}, "period rules"),
    (set(strategy_taxonomy["strategy_id"]), {f"STR{i:02d}" for i in range(1,10)}, "strategy taxonomy"),
    (set(outcome_taxonomy["outcome_id"]), {f"OUT{i:02d}" for i in range(1,19)}, "outcome taxonomy"),
]
for actual, expected, label in expected_sets:
    if actual != expected:
        raise RuntimeError(f"Frozen {label} is incomplete.")

if len(actions) != 22 or len(results) != 69 or len(company_claims) != 6:
    raise RuntimeError("Unexpected Stage 6C cardinalities.")
if len(stage4_findings) != 6:
    raise RuntimeError("Unexpected Stage 4 finding count.")

valid_action_ids = set(actions["action_id"])
valid_result_ids = set(results["result_id"])
valid_claim_ids = set(company_claims["claim_id"])
valid_finding_ids = set(stage4_findings["finding_id"])

print(f"Input checksums passed: {len(stage6d_input_lock)}/{len(stage6d_input_lock)}")
print(f"Stage 6C gate: {final_stage6c['result']} / {final_stage6c['status']}")
print(f"Actions: {len(actions)} | Results: {len(results)} | Claims: {len(company_claims)}")
print(f"Stage 4 findings: {len(stage4_findings)} | Linkage gates: {len(linkage_rules)}")


Input checksums passed: 12/12
Stage 6C gate: PASS_WITH_CAVEAT / passed_with_caveat
Actions: 22 | Results: 69 | Claims: 6
Stage 4 findings: 6 | Linkage gates: 16


## Action-Level Linkage Screening

Screen every Stage 6C action exactly once. A compatible result is assigned only where the entity, geography, period, outcome semantics, and proposed claim are defensible.


In [4]:
screen_specs = {
    "S6CACT_WNG_001": ("","","","brand_product_launch","not_assessable_no_compatible_result","not_assessable","ATTR09","not_assessable",
        "GOLDA launch is documented, but no compatible GOLDA/category result exists.",
        "Describe the documented launch.","Do not infer sales, leadership, reach, or profitability."),
    "S6CACT_WNG_002": ("","","","brand_pack_price","not_assessable_no_compatible_result","not_assessable","ATTR09","not_assessable",
        "The GOLDA pack-price point is documented, but no compatible price/mix, volume, or brand-sales result exists.",
        "Describe the stated pack-price point.","Do not infer pricing effectiveness."),
    "S6CACT_WNG_003": ("","","","brand_distribution","not_assessable_no_compatible_result","not_assessable","ATTR09","not_assessable",
        "GOLDA channel availability is documented, but no compatible distribution result exists.",
        "Describe disclosed channels.","Do not relabel availability as consumer reach or infer sales impact."),
    "S6CACT_WNG_004": ("","","","brand_product_launch","not_assessable_no_compatible_result","not_assessable","ATTR09","not_assessable",
        "ProGuard launch evidence has no compatible brand/category result.",
        "Describe the launch.","Do not infer brand, sales, health, or leadership effects."),
    "S6CACT_WNG_005": ("","","","brand_marketing","not_assessable_no_compatible_result","not_assessable","ATTR09","not_assessable",
        "ProGuard activation is documented, but no compatible outcome is available.",
        "Describe the activation.","Do not infer marketing effectiveness."),
    "S6CACT_WNG_006": ("","","","brand_product_launch","not_assessable_no_compatible_result","not_assessable","ATTR09","not_assessable",
        "Ale-Ale FunFlava launch has no compatible brand/category result.",
        "Describe the launch.","Do not infer leadership, market share, or sales effects."),
    "S6CACT_WNG_007": ("","","","brand_pack_price","not_assessable_no_compatible_result","not_assessable","ATTR09","not_assessable",
        "The Ale-Ale price point has no compatible price/mix, volume, or brand-sales result.",
        "Describe the stated price point.","Do not infer affordability effectiveness or demand response."),
    "S6CACT_WNG_008": ("","","","brand_product_launch","not_assessable_no_compatible_result","not_assessable","ATTR09","not_assessable",
        "ISOPLUS COCO launch has no compatible brand/category result.",
        "Describe the launch.","Do not infer sales, reach, momentum, or category effects."),

    "S6CACT_MYR_001": ("S6CRES_MYR_2025_OUT07","S6CRES_MYR_2024_OUT07","","company_level_mixed_geography",
        "eligible_with_caveat_temporal","temporally_aligned","ATTR06","descriptive_alignment_only",
        "Product innovation overlaps FY2025 consolidated sales, but the action is company-wide, the result mixes domestic/export activity, and no brand-level mechanism is isolated.",
        "Describe the action and FY2025 sales as temporally aligned company-level evidence with caveats.",
        "Do not state that product innovation caused sales performance or Indonesian household demand."),
    "S6CACT_MYR_002": ("S6CRES_MYR_2025_OUT07","S6CRES_MYR_2024_OUT07","S6CCLM_MYR_001","company_level_mixed_geography",
        "company_reported_only","company_reported","ATTR07","not_eligible_for_independent_causal_claim",
        "The pricing policy and FY2025 sales are observable; the company separately reported that raw-material inflation required selling-price adjustments and affected sales-target achievement. The statements are related but not identical.",
        "Report the pricing/sales explanation only as company-reported with mixed-geography caveats.",
        "Do not convert management attribution into an independently established pricing effect."),
    "S6CACT_MYR_003": ("","","","corporate_capital_allocation",
        "not_assessable_no_compatible_result","not_assessable","ATTR09","not_assessable",
        "The buyback is documented but no compatible operational or consumer-market result is attributable to it.",
        "Describe the buyback as capital-allocation context.",
        "Do not treat the buyback as consumer-market success."),
    "S6CACT_MYR_004": ("S6CRES_MYR_2025_OUT13","S6CRES_MYR_2024_OUT13","","company_level_mixed_geography",
        "eligible_with_caveat_temporal","temporally_aligned","ATTR06","descriptive_alignment_only",
        "Domestic-input sourcing overlaps FY2025 operating profit, but input costs are a material alternative factor and no quantified procurement effect is disclosed.",
        "Describe the sourcing action and operating profit as temporally aligned with substantial caveats.",
        "Do not state that domestic sourcing improved or protected operating profit."),

    "S6CACT_UNV_001": ("S6CRES_UNV_ORIG_2024_OUT07","S6CRES_UNV_ORIG_2023_OUT07","","indonesia_company_level",
        "eligible_with_caveat_temporal","temporally_aligned","ATTR06","descriptive_alignment_only",
        "Distribution actions and FY2024 company sales overlap in entity and period, but the result is company-wide and alternative factors are not isolated.",
        "Describe temporal company-level alignment with caveats.",
        "Do not state that distribution changes caused FY2024 sales."),
    "S6CACT_UNV_002": ("S6CRES_UNV_ORIG_2024_OUT07","S6CRES_UNV_ORIG_2023_OUT07","","indonesia_company_level",
        "eligible_with_caveat_temporal","temporally_aligned","ATTR06","descriptive_alignment_only",
        "Standardized promotion overlaps FY2024 company sales, but the outcome is too broad to isolate a promotion effect.",
        "Describe temporal company-level alignment.",
        "Do not infer marketing effectiveness or causation."),
    "S6CACT_UNV_003": ("","","S6CCLM_UNV_001","general_trade_company_claim",
        "company_reported_only_no_observable_result","company_reported","ATTR07","not_assessable_independent",
        "Small-pack/coinage pricing is documented and the company reported that it helped sustain volume, but no compatible OUT09 volume observation exists.",
        "Report the stated volume effect only as company-reported.",
        "Do not present the claimed volume effect as an independently observed result."),
    "S6CACT_UNV_004": ("","","","brand_product_relaunch",
        "not_assessable_no_compatible_result","not_assessable","ATTR09","not_assessable",
        "Wipol relaunch is brand-specific but no Wipol result exists.",
        "Describe the relaunch.","Do not assign company-wide sales/profit to Wipol."),
    "S6CACT_UNV_005": ("","","","brand_product_relaunch",
        "not_assessable_no_compatible_result","not_assessable","ATTR09","not_assessable",
        "Trika relaunch is brand-specific but no Trika result exists.",
        "Describe the relaunch.","Do not assign company-wide sales/profit to Trika."),
    "S6CACT_UNV_006": ("","","","brand_product_launch_execution",
        "not_assessable_no_compatible_result","not_assessable","ATTR09","not_assessable",
        "Sunlight product/packaging execution has no compatible brand-level product-performance result.",
        "Describe the product/packaging action.","Do not infer purchase, sales, or leadership effects."),
    "S6CACT_UNV_007": ("","","","brand_marketing_launch_execution",
        "not_assessable_no_compatible_result","not_assessable","ATTR09","not_assessable",
        "Sunlight promotion has no compatible brand-level marketing result.",
        "Describe the promotional action.","Do not infer campaign effectiveness."),
    "S6CACT_UNV_008": ("S6CRES_UNV_Q1_2025_OUT17","","","brand_distribution_availability",
        "eligible_descriptive_direct","directly_supported","ATTR01","direct_descriptive_link_only",
        "Action and result share Sunlight, Indonesia, Q1 2025, and the same distribution-availability construct; the company reported coverage above 70% of direct stores.",
        "It is directly supported that the launch included broad outlet coverage and the company reported coverage above 70% of direct stores.",
        "Do not claim that outlet coverage caused sales, market share, consumer reach, or financial performance."),
    "S6CACT_UNV_009": ("","","","brand_pricing_launch_execution",
        "not_assessable_no_compatible_result","not_assessable","ATTR09","not_assessable",
        "Sunlight pricing through Net Revenue Management has no compatible OUT10 price/mix or brand-level commercial result.",
        "Describe the pricing action.","Do not infer a pricing or mix effect."),
    "S6CACT_UNV_010": ("S6CRES_UNV_CONT_FY2025_OUT07","S6CRES_UNV_CONT_FY2024_REPRESENTED_OUT07","","ownership_and_reporting_perimeter",
        "context_only_perimeter","context_only","ATTR09","context_only_not_performance_attribution",
        "The Ice Cream separation changes ownership/reporting scope. Continuing-operation FY2025 sales and the re-presented FY2024 comparator support perimeter interpretation, not disposal-effect attribution.",
        "Use the separation to explain continuing-operation and represented-comparator scope.",
        "Do not claim that separation caused the continuing-operation sales level."),
}

if set(screen_specs) != valid_action_ids:
    raise RuntimeError("Screening specification does not cover all actions exactly.")

cols = [
    "screen_id","action_id","canonical_group","strategy_id","action_period",
    "brand_or_business","primary_result_id","comparator_result_id",
    "companion_claim_id","proposed_linkage_scope","screening_status",
    "evidence_status","attribution_rule_id","independent_claim_eligibility",
    "screening_rationale","allowed_claim","prohibited_claim",
]

rows=[]
for i, a in enumerate(actions.itertuples(index=False), 1):
    spec = screen_specs[a.action_id]
    primary, comparator, claim_id = spec[0], spec[1], spec[2]

    if primary and primary not in valid_result_ids:
        raise RuntimeError(f"Unknown primary result: {primary}")
    if comparator and comparator not in valid_result_ids:
        raise RuntimeError(f"Unknown comparator result: {comparator}")
    if claim_id and claim_id not in valid_claim_ids:
        raise RuntimeError(f"Unknown companion claim: {claim_id}")

    rows.append({
        "screen_id": f"S6DLNK{i:03d}",
        "action_id": a.action_id,
        "canonical_group": a.canonical_group,
        "strategy_id": a.strategy_code,
        "action_period": a.action_period,
        "brand_or_business": a.brand_or_business,
        "primary_result_id": primary,
        "comparator_result_id": comparator,
        "companion_claim_id": claim_id,
        "proposed_linkage_scope": spec[3],
        "screening_status": spec[4],
        "evidence_status": spec[5],
        "attribution_rule_id": spec[6],
        "independent_claim_eligibility": spec[7],
        "screening_rationale": spec[8],
        "allowed_claim": spec[9],
        "prohibited_claim": spec[10],
    })

stage6d_action_linkage_screening = pd.DataFrame(rows, columns=cols)

expected_counts = {
    "eligible_descriptive_direct": 1,
    "eligible_with_caveat_temporal": 4,
    "company_reported_only": 1,
    "company_reported_only_no_observable_result": 1,
    "context_only_perimeter": 1,
    "not_assessable_no_compatible_result": 14,
}
actual_counts = stage6d_action_linkage_screening["screening_status"].value_counts().to_dict()
if actual_counts != expected_counts:
    raise RuntimeError(f"Unexpected screening counts: {actual_counts}")

print("Action-level screening complete.")
print(stage6d_action_linkage_screening["screening_status"].value_counts())


Action-level screening complete.
screening_status
not_assessable_no_compatible_result           14
eligible_with_caveat_temporal                  4
company_reported_only                          1
company_reported_only_no_observable_result     1
eligible_descriptive_direct                    1
context_only_perimeter                         1
Name: count, dtype: int64


## Gate-Level Screening Matrix

Evaluate every action against all 16 frozen linkage gates. Conditional gates may be not applicable; missing outcomes stay explicit.


In [5]:
rule_lookup = linkage_rules.set_index("linkage_rule_id").to_dict("index")

mayora_result_actions = {"S6CACT_MYR_001","S6CACT_MYR_002","S6CACT_MYR_004"}
unilever_company_result_actions = {"S6CACT_UNV_001","S6CACT_UNV_002"}
brand_no_result_actions = {
    "S6CACT_WNG_001","S6CACT_WNG_002","S6CACT_WNG_003","S6CACT_WNG_004",
    "S6CACT_WNG_005","S6CACT_WNG_006","S6CACT_WNG_007","S6CACT_WNG_008",
    "S6CACT_UNV_004","S6CACT_UNV_005","S6CACT_UNV_006","S6CACT_UNV_007",
    "S6CACT_UNV_009",
}
claim_actions = {"S6CACT_MYR_002","S6CACT_UNV_003"}

gate_rows=[]

for s in stage6d_action_linkage_screening.itertuples(index=False):
    has_result = bool(s.primary_result_id)
    for rule in linkage_rules.itertuples(index=False):
        rid = rule.linkage_rule_id

        if rid == "LNK01":
            gr, gi = "met", "A documented period-bounded action exists."
        elif rid == "LNK02":
            gr, gi = "met", "Stage 6C retained the action within the valid ownership perimeter."
        elif rid == "LNK03":
            if not has_result:
                gr, gi = "not_met", "No compatible result exists for entity alignment."
            elif s.action_id == "S6CACT_UNV_010":
                gr, gi = "limited", "Entity alignment is valid for perimeter interpretation, not performance attribution."
            else:
                gr, gi = "met", "Action and primary result share a defensible company/brand perimeter."
        elif rid == "LNK04":
            if not has_result:
                gr, gi = "not_assessable", "No result exists for geography alignment."
            elif s.action_id in mayora_result_actions:
                gr, gi = "limited", "Both items include domestic/export activity; no Indonesia-household-demand claim is allowed."
            elif s.action_id in unilever_company_result_actions or s.action_id == "S6CACT_UNV_010":
                gr, gi = "limited", "Company result includes export activity where reported."
            else:
                gr, gi = "met", "Action and result share Indonesia scope."
        elif rid == "LNK05":
            if s.action_id == "S6CACT_UNV_008":
                gr, gi = "met", "Sunlight scope matches the launch-distribution result."
            elif s.action_id in brand_no_result_actions:
                gr, gi = "not_met", "No compatible brand/category result is available."
            elif s.action_id == "S6CACT_UNV_003":
                gr, gi = "not_assessable", "The claimed General Trade volume result is not observed as OUT09."
            elif s.action_id == "S6CACT_UNV_010":
                gr, gi = "not_met", "Ice Cream business scope does not match company continuing-operation sales for performance attribution."
            else:
                gr, gi = "not_applicable", "The proposed screening is company-level."
        elif rid == "LNK06":
            if not has_result:
                gr, gi = "not_assessable", "Temporal linkage cannot be assessed without a result."
            elif s.action_id == "S6CACT_UNV_008":
                gr, gi = "met", "Action and result share Q1 2025 launch timing."
            elif s.action_id == "S6CACT_UNV_010":
                gr, gi = "limited", "The December separation is valid for perimeter context, not full-year performance attribution."
            else:
                gr, gi = "met_with_overlap_caveat", "Action and result share the annual period; within-year ordering is not isolated."
        elif rid == "LNK07":
            gr, gi = "not_applicable", "No lagged linkage is proposed."
        elif rid == "LNK08":
            if has_result:
                gr, gi = "met", "Primary result retains source-native metric, unit, period, and scope."
            else:
                gr, gi = "not_met", "No compatible outcome observation is available."
        elif rid == "LNK09":
            if s.comparator_result_id:
                gr, gi = "met", "A compatible within-scope comparator is retained."
            else:
                gr, gi = "not_applicable", "No change-from-baseline claim is proposed."
        elif rid == "LNK10":
            if s.action_id in mayora_result_actions:
                gr, gi = "met_with_caveat", "Raw-material/input-cost pressure is retained as an alternative factor."
            elif s.action_id in unilever_company_result_actions:
                gr, gi = "not_met_for_interpretation", "Material alternative factors are not fully isolated for company-sales interpretation."
            elif s.action_id == "S6CACT_UNV_008":
                gr, gi = "not_applicable_descriptive_measurement", "Allowed claim is limited to the reported distribution measure."
            elif s.action_id == "S6CACT_UNV_010":
                gr, gi = "not_applicable_perimeter_context", "Screening concerns reporting perimeter, not effectiveness."
            else:
                gr, gi = "not_applicable_without_result", "No performance interpretation is attempted."
        elif rid == "LNK11":
            if s.action_id in claim_actions:
                gr, gi = "met", "Related management explanation remains explicitly company_reported."
            else:
                gr, gi = "not_applicable", "No company-attributed explanation is used."
        elif rid == "LNK12":
            gr, gi = "met", "Missing results remain missing and are never zero-filled."
        elif rid == "LNK13":
            gr, gi = "not_applicable_within_company_only", "No cross-group strategy-effect ranking is performed."
        elif rid == "LNK14":
            gr, gi = "met", "Disclosure coverage is not interpreted as company performance."
        elif rid == "LNK15":
            gr, gi = "met", "Allowed claims remain descriptive, contextual, or company-reported."
        elif rid == "LNK16":
            gr, gi = "met", "No action evidence becomes a score, weight, or overall winner."
        else:
            raise RuntimeError(f"Unhandled rule {rid}")

        gate_rows.append({
            "screen_id": s.screen_id,
            "action_id": s.action_id,
            "linkage_rule_id": rid,
            "eligibility_gate": rule.eligibility_gate,
            "gate_type": rule.gate_type,
            "gate_result": gr,
            "gate_interpretation": gi,
        })

stage6d_linkage_gate_results = pd.DataFrame(gate_rows)

if len(stage6d_linkage_gate_results) != 352:
    raise RuntimeError("Expected 352 action-gate rows.")

per_action = stage6d_linkage_gate_results.groupby("action_id")["linkage_rule_id"].nunique()
if not (per_action == 16).all():
    raise RuntimeError("Every action must have all 16 linkage gates.")

print(f"Gate-level screening rows: {len(stage6d_linkage_gate_results)}")
print(stage6d_linkage_gate_results["gate_result"].value_counts())


Gate-level screening rows: 352
gate_result
met                                       156
not_applicable                             64
not_met                                    44
not_assessable                             31
not_applicable_within_company_only         22
not_applicable_without_result              15
limited                                     8
met_with_overlap_caveat                     5
met_with_caveat                             3
not_met_for_interpretation                  2
not_applicable_descriptive_measurement      1
not_applicable_perimeter_context            1
Name: count, dtype: int64


## Company Claims, Stage 4 Context, and Screening Summary

Keep all six management-attribution claims separate from independent evidence, screen inherited Stage 4 portfolio findings as context, summarize all focal groups, and register unresolved caveats.


In [6]:
claim_specs = {
    "S6CCLM_IDF_001": ("","S6CRES_IDF_2024_OUT07;S6CRES_IDF_2024_OUT13",
        "no_discrete_documented_action","observed_results_available","company_reported_only",
        "not_eligible_for_independent_claim",
        "FY2024 sales and operating profit are observed, but no discrete Indofood action matches the claimed integrated-model factor."),
    "S6CCLM_IDF_002": ("","S6CRES_IDF_2025_OUT07;S6CRES_IDF_2025_OUT13",
        "no_discrete_documented_action","observed_results_available","company_reported_only",
        "not_eligible_for_independent_claim",
        "FY2025 sales and operating profit are observed, but the integrated-model explanation remains management attribution."),
    "S6CCLM_ICBP_001": ("","S6CRES_ICBP_2024_OUT07;S6CRES_ICBP_2024_OUT13",
        "no_discrete_documented_action","observed_results_available","company_reported_only",
        "not_eligible_for_independent_claim",
        "ICBP FY2024 sales and operating profit are observed, but higher volume/productivity remains management attribution."),
    "S6CCLM_MYR_001": ("S6CACT_MYR_002","S6CRES_MYR_2025_OUT07",
        "related_strategy_action_with_wording_caveat","observed_result_available",
        "company_reported_with_action_correspondence_caveat",
        "not_eligible_for_independent_causal_claim",
        "The STR02 action and selling-price-adjustment explanation are related but not identical; sales is mixed-geography."),
    "S6CCLM_MYR_002": ("","S6CRES_MYR_2025_OUT13",
        "external_factor_not_strategy_action","observed_result_available",
        "company_reported_alternative_factor","not_eligible_as_strategy_effect",
        "Raw-material cost pressure explains operating-profit performance as an alternative factor, not a strategy effect."),
    "S6CCLM_UNV_001": ("S6CACT_UNV_003","",
        "documented_related_action","no_compatible_out09_observation",
        "company_reported_without_observable_result","not_assessable_independent",
        "Small-pack pricing is documented, but no source-native OUT09 volume observation supports an independent check."),
}

claim_rows=[]
for c in company_claims.itertuples(index=False):
    spec = claim_specs[c.claim_id]
    action_ids = [x for x in spec[0].split(";") if x]
    result_ids = [x for x in spec[1].split(";") if x]
    if set(action_ids) - valid_action_ids or set(result_ids) - valid_result_ids:
        raise RuntimeError(f"Invalid company-claim linkage IDs for {c.claim_id}.")
    claim_rows.append({
        "claim_id": c.claim_id,
        "canonical_group": c.canonical_group,
        "claim_period": c.claim_period,
        "related_action_ids": spec[0],
        "matched_result_ids": spec[1],
        "matched_result_count": len(result_ids),
        "action_correspondence": spec[2],
        "result_availability": spec[3],
        "screening_status": spec[4],
        "evidence_status": "company_reported",
        "independent_claim_eligibility": spec[5],
        "screening_rationale": spec[6],
    })

stage6d_company_claim_screening = pd.DataFrame(claim_rows)
if len(stage6d_company_claim_screening) != 6:
    raise RuntimeError("Expected 6 company-claim screening rows.")

portfolio_specs = [
    ("Wings Group",";".join([f"S6CACT_WNG_{i:03d}" for i in range(1,9)]),"FND4_02;FND4_03","context_only",
     "Use leadership/stability findings as group-level context.",
     "Do not attribute them to GOLDA, ProGuard, Ale-Ale, or ISOPLUS actions.",
     "Stage 4 evidence is broader than the brand-specific Wings actions and no compatible outcomes exist."),
    ("Indofood","","FND4_01;FND4_03;FND4_04","not_assessable_action_absent",
     "Retain breadth, momentum, and persistence findings independently.",
     "Do not infer which Indofood strategy caused those findings.",
     "Stage 6C contains company explanations but no discrete Indofood/ICBP action for direct Stage 4 linkage."),
    ("Mayora",";".join([f"S6CACT_MYR_{i:03d}" for i in range(1,5)]),"FND4_05;FND4_06","context_only",
     "Use ownership-sensitivity and overall-winner findings as governance context.",
     "Do not use 2025 actions to explain ownership sensitivity or alter strict-control scope.",
     "Inherited Mayora findings do not directly match the Stage 6C action outcomes."),
    ("Unilever Indonesia",";".join([f"S6CACT_UNV_{i:03d}" for i in range(1,11)]),"FND4_02;FND4_03;FND4_04","context_only",
     "Use leadership, momentum, and persistence as portfolio context.",
     "Do not attribute them to Wipol, Trika, Sunlight, channel, pricing, promotion, or disposal actions.",
     "Stage 4 granularity differs and the 2026 snapshot cannot be bridged to the 2022–2025 strategy window."),
]

portfolio_rows=[]
for group, action_ids, finding_ids, status, allowed, prohibited, rationale in portfolio_specs:
    if set([x for x in action_ids.split(";") if x]) - valid_action_ids:
        raise RuntimeError(f"Invalid action IDs in portfolio screening: {group}")
    if set([x for x in finding_ids.split(";") if x]) - valid_finding_ids:
        raise RuntimeError(f"Invalid Stage 4 finding IDs: {group}")
    portfolio_rows.append({
        "canonical_group": group,
        "stage6c_action_ids": action_ids,
        "stage4_finding_ids": finding_ids,
        "screening_status": status,
        "allowed_use": allowed,
        "prohibited_inference": prohibited,
        "screening_rationale": rationale,
    })

stage6d_portfolio_context_screening = pd.DataFrame(portfolio_rows)

focal_groups = ["Wings Group","Indofood","Mayora","Unilever Indonesia"]
portfolio_status = stage6d_portfolio_context_screening.set_index("canonical_group")["screening_status"].to_dict()
status_cols = {
    "eligible_descriptive_direct":"directly_supported_count",
    "eligible_with_caveat_temporal":"temporally_aligned_count",
    "company_reported_only":"company_reported_count",
    "company_reported_only_no_observable_result":"company_reported_count",
    "context_only_perimeter":"context_only_count",
    "not_assessable_no_compatible_result":"not_assessable_count",
}
summary=[]
for group in focal_groups:
    g = stage6d_action_linkage_screening[
        stage6d_action_linkage_screening["canonical_group"] == group
    ]
    row = {
        "canonical_group": group,
        "total_documented_actions": len(g),
        "directly_supported_count": 0,
        "temporally_aligned_count": 0,
        "company_reported_count": 0,
        "context_only_count": 0,
        "not_assessable_count": 0,
        "company_claim_rows": len(stage6d_company_claim_screening[
            stage6d_company_claim_screening["canonical_group"] == group
        ]),
        "portfolio_context_status": portfolio_status[group],
    }
    for status, count in g["screening_status"].value_counts().items():
        row[status_cols[status]] += int(count)
    summary.append(row)

total = {
    "canonical_group":"All focal groups",
    "total_documented_actions":sum(x["total_documented_actions"] for x in summary),
    "directly_supported_count":sum(x["directly_supported_count"] for x in summary),
    "temporally_aligned_count":sum(x["temporally_aligned_count"] for x in summary),
    "company_reported_count":sum(x["company_reported_count"] for x in summary),
    "context_only_count":sum(x["context_only_count"] for x in summary),
    "not_assessable_count":sum(x["not_assessable_count"] for x in summary),
    "company_claim_rows":sum(x["company_claim_rows"] for x in summary),
    "portfolio_context_status":"mixed_context_only_and_not_assessable",
}
summary.append(total)
stage6d_screening_summary = pd.DataFrame(summary)

expected_total = {
    "total_documented_actions":22,
    "directly_supported_count":1,
    "temporally_aligned_count":4,
    "company_reported_count":2,
    "context_only_count":1,
    "not_assessable_count":14,
    "company_claim_rows":6,
}
for col, val in expected_total.items():
    if int(total[col]) != val:
        raise RuntimeError(f"Summary mismatch for {col}: {total[col]} vs {val}")

exception_rows = [
    ("S6DEX001","Wings Group","no_compatible_result_observations","caveat","open",
     "Eight documented Wings actions have no compatible Stage 6C result for action-level linkage.",
     "Retain as evidence/disclosure limitation, not weak performance."),
    ("S6DEX002","Wings Group","private_disclosure_asymmetry","caveat","controlled",
     "Public financial disclosure is materially lower for Wings than for listed focal companies.",
     "Do not rank companies by assessable-link count."),
    ("S6DEX003","Indofood","company_claim_without_discrete_action","caveat","open",
     "Indofood management explanations align to observed results without a matching discrete action.",
     "Keep company-reported and ineligible for an independent strategy-effect claim."),
    ("S6DEX004","Indofood","icbp_claim_without_discrete_action","caveat","open",
     "ICBP volume/productivity/efficiency factors are management attribution rather than a separately extracted action.",
     "Do not upgrade beyond company_reported."),
    ("S6DEX005","Mayora","mixed_geography","caveat","controlled",
     "Mayora action and result evidence mixes domestic/export or international activity.",
     "Use company-level mixed-geography interpretation only."),
    ("S6DEX006","Mayora","pricing_claim_wording_mismatch","caveat","controlled",
     "Competitive-pricing policy and selling-price-adjustment explanation are related but non-identical STR02 evidence.",
     "Retain company_reported status and wording distinction."),
    ("S6DEX007","Mayora","capital_allocation_not_consumer_outcome","informational","controlled",
     "The share buyback has no compatible consumer/operational result.",
     "Keep as capital-allocation context only."),
    ("S6DEX008","Unilever Indonesia","brand_result_granularity_gap","caveat","open",
     "Wipol, Trika, and several Sunlight actions lack compatible brand-level results.",
     "Do not assign company-wide sales/profit to individual brand actions."),
    ("S6DEX009","Unilever Indonesia","interim_distribution_measure","caveat","controlled",
     "The strongest direct pair is a Q1 2025 launch-distribution measure, not a full-year commercial outcome.",
     "Use only as direct support for launch distribution availability."),
    ("S6DEX010","Unilever Indonesia","represented_comparator_boundary","caveat","controlled",
     "FY2024 original and re-presented continuing-operation comparators remain different scopes.",
     "Use represented comparator only with FY2025 continuing-operation evidence."),
    ("S6DEX011","Cross-company context","stage4_granularity_mismatch","caveat","controlled",
     "Stage 4 portfolio findings are broader than most Stage 6C actions and include a separate 2026 snapshot.",
     "Use as portfolio context unless brand/category/period alignment is explicit."),
    ("S6DEX012","Cross-company context","cross_group_effect_comparability","caveat","controlled",
     "Linkage statuses differ partly because disclosure, entity scope, geography, and result granularity differ.",
     "Do not rank strategy effectiveness or revise the overall-winner conclusion from screening counts."),
]
stage6d_screening_exceptions = pd.DataFrame(
    exception_rows,
    columns=["exception_id","canonical_group","exception_type","severity","status","description","required_treatment"]
)

print("Company claims:", len(stage6d_company_claim_screening))
print("Portfolio context rows:", len(stage6d_portfolio_context_screening))
print("Exceptions:", len(stage6d_screening_exceptions))
display(stage6d_screening_summary)


Company claims: 6
Portfolio context rows: 4
Exceptions: 12


,canonical_group,total_documented_actions,directly_supported_count,temporally_aligned_count,company_reported_count,context_only_count,not_assessable_count,company_claim_rows,portfolio_context_status
0,Wings Group,8,0,0,0,0,8,0,context_only
1,Indofood,0,0,0,0,0,0,3,not_assessable_action_absent
2,Mayora,4,0,2,1,0,1,2,context_only
3,Unilever Indonesia,10,1,2,1,1,5,1,context_only
4,All focal groups,22,1,4,2,1,14,6,mixed_context_only_and_not_assessable


## Stage 6D Validation and Canonical Outputs

Validate coverage and evidence boundaries, write the eight canonical Stage 6D outputs, build the manifest, and run final QA.


In [7]:
validation_rows = [
("S6D001","input_integrity","All 12 governed Stage 4–6C inputs match locked SHA-256 values.","12/12 inputs passed","passed","no","Stop Stage 6D if any input differs."),
("S6D002","prior_stage_gate","Stage 6C final gate remains PASS_WITH_CAVEAT.","PASS_WITH_CAVEAT","passed_with_caveat","no","Carry Stage 6C caveats forward."),
("S6D003","action_universe","All 22 Stage 6C actions are screened exactly once.","22/22 actions","passed","no","Do not select only favorable actions."),
("S6D004","gate_registry","Frozen linkage registry contains LNK01–LNK16.","16/16 gates","passed","no","Use all pre-specified gates."),
("S6D005","gate_matrix","Every action is evaluated against all 16 gates.","352/352 rows","passed","no","Do not omit failed/conditional gates."),
("S6D006","strategy_taxonomy","All strategy IDs remain within STR01–STR09.","valid","passed","no","Reject post-hoc classes."),
("S6D007","result_provenance","All nonblank result IDs exist in Stage 6C.","valid","passed","no","Do not manufacture outcomes."),
("S6D008","claim_provenance","All nonblank companion claims exist in Stage 6C.","valid","passed","no","Keep company claims separate."),
("S6D009","direct_support_count","Exactly one action-result pair meets direct descriptive screening.","1","passed_with_caveat","no","Do not interpret the count as effectiveness."),
("S6D010","direct_support_scope","Direct pair is Sunlight Q1 2025 distribution availability.","UNV_008 -> OUT17","passed","no","Limit claim to reported outlet coverage."),
("S6D011","temporal_alignment","Four pairs are temporal alignment only.","4","passed_with_caveat","no","Do not upgrade to causation."),
("S6D012","company_reported_action_screening","Two action screenings depend on company-reported explanations.","2","passed_with_caveat","no","Keep ATTR07/company_reported."),
("S6D013","context_only_perimeter","One ownership action is reporting-perimeter context only.","1","passed","no","Do not treat disposal as a performance cause."),
("S6D014","not_assessable","Fourteen actions lack a compatible result.","14","passed_with_caveat","no","Missing compatible results are valid limitations."),
("S6D015","company_claim_universe","All six company-attribution claims are screened.","6/6","passed","no","Do not omit inconvenient claims."),
("S6D016","company_claim_status","All six remain company_reported.","6/6","passed","no","No independent causal upgrade."),
("S6D017","indofood_claim_boundary","Indofood/ICBP claims lack matching discrete actions.","3 claims constrained","passed_with_caveat","no","Report as company-reported only."),
("S6D018","mayora_pricing_boundary","Mayora pricing action and adjustment claim remain related but non-identical.","retained","passed_with_caveat","no","Do not collapse wording."),
("S6D019","unilever_volume_boundary","Unilever small-pack volume claim lacks OUT09 observation.","not assessable independently","passed_with_caveat","no","Keep claimed effect company-reported."),
("S6D020","stage4_context_coverage","All four focal groups receive Stage 4 context screening.","4/4","passed","no","Do not silently upgrade group findings."),
("S6D021","stage4_snapshot_boundary","2026 snapshot is not bridged to 2022–2025 strategy window.","retained","passed","no","Keep 2026 separate."),
("S6D022","wings_disclosure_neutrality","Wings not-assessable rows are not interpreted as weak performance.","retained","passed_with_caveat","no","Do not rank by assessable-link count."),
("S6D023","mayora_geography","Mayora candidate links retain mixed-geography caveats.","retained","passed_with_caveat","no","Do not infer Indonesian household demand."),
("S6D024","unilever_brand_alignment","Brand actions without brand outcomes are not linked to company-wide results.","retained","passed","no","Preserve granularity."),
("S6D025","distribution_semantics","Direct-store coverage remains distribution availability, not consumer reach.","retained","passed","no","Do not relabel metric."),
("S6D026","unilever_comparator_scope","FY2025 continuing-operation screening uses re-presented FY2024 comparator.","retained","passed","no","Keep original and represented observations separate."),
("S6D027","market_share_boundary","No market-share outcome is inferred.","0 inferred links","passed","no","Require explicit market-share measurement."),
("S6D028","missingness","Missing outcomes stay blank, not zero-filled.","preserved","passed","no","Do not impute disclosure gaps."),
("S6D029","lag_policy","No post-hoc lagged linkage is introduced.","0 lagged links","passed","no","Later lag requires pre-specified rationale."),
("S6D030","alternative_factors","Mayora raw-material cost pressure remains an alternative factor.","retained","passed_with_caveat","no","Do not over-attribute pricing/sourcing."),
("S6D031","cross_company_ranking","No linkage count becomes a cross-company strategy-effect ranking.","none","passed","no","Use counts as evidence diagnostics only."),
("S6D032","causal_boundary","No linkage is labelled causal.","none upgraded","passed","no","Causation requires identification design."),
("S6D033","composite_boundary","No strategy score or weighting is created.","none","passed","no","Keep dimension/case level."),
("S6D034","overall_winner_boundary","Stage 4 overall-winner conclusion is unchanged.","unchanged","passed","no","Revision requires comparable pre-specified framework."),
("S6D035","research_question_alignment","Screening addresses SRQ03–SRQ10 and preserves SRQ11 governance.","covered","passed","no","Do not treat screening as final synthesis."),
("S6D036","exception_registry","Twelve screening caveats are registered.","12","passed_with_caveat","no","Carry limitations forward."),
("S6D037","reporting_boundary","No report or README is created.","none","passed","no","Defer reporting."),
("S6D038","raw_source_boundary","No raw copyrighted source is written.","none","passed","no","Keep reference-only source treatment."),
("S6D039","screening_summary","Summary reconciles to 22 actions and six claims.","reconciled","passed","no","Use summary only for diagnostics."),
("S6D040","stage_gate","Linkage eligibility screening is complete enough for bounded interpretation and synthesis.","PASS_WITH_CAVEAT","passed_with_caveat","no","Carry disclosure asymmetry, mixed geography, company-claim labels, missing outcomes, Stage 4 granularity limits, and non-causal boundaries into Stage 6E."),
]
stage6d_linkage_validation = pd.DataFrame(
    validation_rows,
    columns=["check_id","validation_area","check_description","result","status","critical_failure","required_treatment"]
)

if set(stage6d_linkage_validation["check_id"]) != {f"S6D{i:03d}" for i in range(1,41)}:
    raise RuntimeError("Validation registry incomplete.")
if (stage6d_linkage_validation["critical_failure"] == "yes").any():
    raise RuntimeError("Critical Stage 6D validation failure.")

direct = stage6d_action_linkage_screening[
    stage6d_action_linkage_screening["screening_status"] == "eligible_descriptive_direct"
]
if len(direct) != 1:
    raise RuntimeError("Expected exactly one direct descriptive linkage.")
if (
    direct.iloc[0]["action_id"] != "S6CACT_UNV_008"
    or direct.iloc[0]["primary_result_id"] != "S6CRES_UNV_Q1_2025_OUT17"
):
    raise RuntimeError("Unexpected direct descriptive pair.")
if set(stage6d_company_claim_screening["evidence_status"]) != {"company_reported"}:
    raise RuntimeError("Company claims were upgraded.")

final_stage6d = stage6d_linkage_validation[
    stage6d_linkage_validation["check_id"] == "S6D040"
].iloc[0]
if (
    final_stage6d["result"] != "PASS_WITH_CAVEAT"
    or final_stage6d["status"] != "passed_with_caveat"
):
    raise RuntimeError("Unexpected Stage 6D gate.")

output_frames = {
    "metadata/stage6d_input_lock.csv": stage6d_input_lock,
    "data/analytical/stage6d_action_linkage_screening.csv": stage6d_action_linkage_screening,
    "data/analytical/stage6d_linkage_gate_results.csv": stage6d_linkage_gate_results,
    "data/analytical/stage6d_company_claim_screening.csv": stage6d_company_claim_screening,
    "data/analytical/stage6d_portfolio_context_screening.csv": stage6d_portfolio_context_screening,
    "data/analytical/stage6d_screening_summary.csv": stage6d_screening_summary,
    "metadata/stage6d_screening_exceptions.csv": stage6d_screening_exceptions,
    "metadata/stage6d_linkage_validation.csv": stage6d_linkage_validation,
}

for path, df in output_frames.items():
    dest = OUTPUT_ROOT / path
    dest.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(dest, index=False, encoding="utf-8")

manifest=[]
for path, df in output_frames.items():
    manifest.append({
        "file_path":path,
        "artifact_type":"csv",
        "row_count":len(df),
        "sha256":sha256_file(OUTPUT_ROOT/path),
        "locked_input_commit":INPUT_COMMIT,
    })
stage6d_output_manifest = pd.DataFrame(manifest)
manifest_path = OUTPUT_ROOT / "metadata/stage6d_output_manifest.csv"
stage6d_output_manifest.to_csv(manifest_path, index=False, encoding="utf-8")

expected_rows = {
    "metadata/stage6d_input_lock.csv":12,
    "data/analytical/stage6d_action_linkage_screening.csv":22,
    "data/analytical/stage6d_linkage_gate_results.csv":352,
    "data/analytical/stage6d_company_claim_screening.csv":6,
    "data/analytical/stage6d_portfolio_context_screening.csv":4,
    "data/analytical/stage6d_screening_summary.csv":5,
    "metadata/stage6d_screening_exceptions.csv":12,
    "metadata/stage6d_linkage_validation.csv":40,
}
for path, expected in expected_rows.items():
    got = len(pd.read_csv(OUTPUT_ROOT/path, dtype=str, keep_default_na=False))
    if got != expected:
        raise RuntimeError(f"{path}: expected {expected}, found {got}")

manifest_check = pd.read_csv(manifest_path, dtype=str, keep_default_na=False)
if len(manifest_check) != 8:
    raise RuntimeError("Expected 8 manifest rows.")
for row in manifest_check.itertuples(index=False):
    if sha256_file(OUTPUT_ROOT/row.file_path) != row.sha256:
        raise RuntimeError(f"Manifest hash mismatch: {row.file_path}")

binary_ext = {".pdf",".doc",".docx",".xls",".xlsx",".ppt",".pptx",".jpg",".jpeg",".png",".webp",".zip"}
unexpected = [
    str(p) for p in OUTPUT_ROOT.rglob("*")
    if p.is_file() and p.suffix.lower() in binary_ext
]
if unexpected:
    raise RuntimeError(f"Unexpected binary/raw files: {unexpected}")

print("Stage 6D final QA passed.")
print("Locked inputs: 12")
print("Action screenings: 22")
print("Gate rows: 352")
print("Company claims: 6")
print("Portfolio-context rows: 4")
print("Summary rows: 5")
print("Exceptions: 12")
print("Validation checks: 40")
print("Critical failures: 0")
print("Direct descriptive links: 1")
print("Temporally aligned links: 4")
print("Company-reported action screenings: 2")
print("Context-only perimeter links: 1")
print("Not-assessable actions: 14")
print("Raw copyrighted source files written: 0")
print("Cross-company strategy-effect ranking created: 0")
print("Causal effects estimated: 0")
print("Manifest hashes: 8/8 matched")
print(f"Stage 6D gate: {final_stage6d['result']} / {final_stage6d['status']}")
display(stage6d_screening_summary)


Stage 6D final QA passed.
Locked inputs: 12
Action screenings: 22
Gate rows: 352
Company claims: 6
Portfolio-context rows: 4
Summary rows: 5
Exceptions: 12
Validation checks: 40
Critical failures: 0
Direct descriptive links: 1
Temporally aligned links: 4
Company-reported action screenings: 2
Context-only perimeter links: 1
Not-assessable actions: 14
Raw copyrighted source files written: 0
Cross-company strategy-effect ranking created: 0
Causal effects estimated: 0
Manifest hashes: 8/8 matched
Stage 6D gate: PASS_WITH_CAVEAT / passed_with_caveat


,canonical_group,total_documented_actions,directly_supported_count,temporally_aligned_count,company_reported_count,context_only_count,not_assessable_count,company_claim_rows,portfolio_context_status
0,Wings Group,8,0,0,0,0,8,0,context_only
1,Indofood,0,0,0,0,0,0,3,not_assessable_action_absent
2,Mayora,4,0,2,1,0,1,2,context_only
3,Unilever Indonesia,10,1,2,1,1,5,1,context_only
4,All focal groups,22,1,4,2,1,14,6,mixed_context_only_and_not_assessable
